In [1]:
from rag.retriever import query_retriever, generate_augmented_prompt, llm_model, vectorstore_initializer, test_query_retriever
from rag.ingestion import embedding_model
from datasets import Dataset




C:\Users\Dell\PycharmProjects\GenAi Complete Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
test_set = [
    {
        "question": "According to David Lay Williams and Alan J. Kellner, what specific category of rulers does Lord Voldemort perfectly fit into?",
        "ground_truth": "Voldemort fits perfectly into Plato's category of 'least trustworthy rulers.'"
    },
    {
        "question": "What are the four less-desirable forms of government that Plato describes in addition to his ideal society?",
        "ground_truth": "The four less-desirable governments are timocracy (rule by those motivated by honor), oligarchy (rule by a small group, usually the wealthy), democracy (rule by the common people), and tyranny (rule by a tyrant)."
    },
    {
        "question": "How do Tom Riddle's childhood actions in the orphanage mirror Plato's description of tyrannical men in their early stages?",
        "ground_truth": "Plato describes early-stage tyrannical men as those who steal, commit burglary, snatch purses, rob travelers, defile temples, and enslave people. This finds a parallel in Tom Riddle's childhood at the orphanage, where he steals from other children and commits escalating acts of cruelty toward animals and other children."
    },
    {
        "question": "According to Plato's 'Myth of Metals' in the Republic, what metals correspond to the souls of the class best suited to rule and protect the city?",
        "ground_truth": "Those with gold or silver in their souls belong in the guardian class; gold souls are best suited to rule, while silver souls protect and defend the city."
    },
    {
        "question": "In the Symposium, what does Plato state is the only path to true immortality, and how must it be accomplished?",
        "ground_truth": "The ascent to the Beautiful itself is the only path to true immortality, and it must be accomplished 'in the proper sequence and in the correct manner.'"
    }
]


In [3]:


results = []

for test in test_set:
    question = test["question"]
    ground_truth = test["ground_truth"]

    # Simulate retrieval and response generation
    vectorstore = vectorstore_initializer(embedding_model())  # Mocked vector store initialization
    retrieved_docs = test_query_retriever(vectorstore, question)  # Mocked retrieval

    augmented_prompt = generate_augmented_prompt(retrieved_docs, question)
    model_response = llm_model().invoke(augmented_prompt)  # Mocked response generation

    # I need question,answer,contexts[],ground_truth
    query_retriever_data = {
        "question": question,
        "answer": model_response.content,
        "contexts": [doc.page_content for doc in retrieved_docs],
        "ground_truth": ground_truth
    }

    results.append(query_retriever_data)


    print(f"Question: {question}")
    print(f"Ground Truth: {ground_truth}")
    print(f"Model Response: {model_response.content}")
    print("-" * 50)



C:\Users\Dell\PycharmProjects\GenAi Complete Code\RAG Projects\Project 7\rag\retriever.py:28: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=chroma_path, embedding_function=embedding_model)


Question: According to David Lay Williams and Alan J. Kellner, what specific category of rulers does Lord Voldemort perfectly fit into?
Ground Truth: Voldemort fits perfectly into Plato's category of 'least trustworthy rulers.'
Model Response: Lord Voldemort perfectly fits Plato’s category of “least trustworthy rulers,” according to David Lay Williams and Alan J. Kellner.
--------------------------------------------------
Question: What are the four less-desirable forms of government that Plato describes in addition to his ideal society?
Ground Truth: The four less-desirable governments are timocracy (rule by those motivated by honor), oligarchy (rule by a small group, usually the wealthy), democracy (rule by the common people), and tyranny (rule by a tyrant).
Model Response: Plato’s four less‑desirable governments are timocracy, oligarchy, democracy, and tyranny.
--------------------------------------------------
Question: How do Tom Riddle's childhood actions in the orphanage mirror 

In [7]:

dataset = Dataset.from_list(results)
dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})

In [15]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

C:\Users\Dell\AppData\Local\Temp\ipykernel_32000\3303913913.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\Dell\AppData\Local\Temp\ipykernel_32000\3303913913.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [16]:
ragas_llm = LangchainLLMWrapper(llm_model())
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model())


C:\Users\Dell\AppData\Local\Temp\ipykernel_32000\2799737882.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm_model())
C:\Users\Dell\AppData\Local\Temp\ipykernel_32000\2799737882.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model())


In [17]:
# evaluate
score = evaluate(
  dataset,
  metrics=[faithfulness, answer_relevancy],
  llm=ragas_llm,
  embeddings=ragas_embeddings
)

print(score)


Evaluating: 100%|██████████| 10/10 [03:00<00:00, 18.00s/it]


{'faithfulness': 1.0000, 'answer_relevancy': 0.7768}
